# Feature generation and holdout-based feature selection

This notebook implements the molecular feature-engineering workflow for the fixed train/validation/test split named by `SPLIT_SCHEME`. The validation set is used only to select the number of features, K. The test set is not used for feature filtering, mutual-information ranking, K selection, or model selection; it is only transformed and saved for later evaluation.

Workflow:

1. Generate (or load cached) molecular representations: RDKit 2D descriptors, MACCS keys, ECFP4 bits, and ChemBERTa-2 MTR embeddings.
2. Add the precomputed graph-neural-network embeddings.
3. Apply the missing-value policy.
4. Remove constant or near-constant features, with separate thresholds for binary and continuous blocks.
5. Apply a target-guided Pearson correlation filter separately to the binary and continuous blocks: among features correlated above the threshold, the one with the larger absolute correlation with MP is kept.
6. Rank the retained features using mutual information with MP.
7. Evaluate candidate Top-K feature sets, including all retained features, with LightGBM on the validation set.
8. Save the selected feature set for downstream model development; keep the test set untouched.

Representations depend only on the SMILES, so they are cached once per compound in `outputs/representation_cache/` and shared by every split; only compounds missing from the cache are computed.

The graph embeddings (`0_data/outputs/combined_data_graph_embeddings.csv.gz`) were generated outside this repository by an external graph model that was not trained with MP labels, so they do not depend on this split. `0_data/2_stereo_cleanup.ipynb` restricts them to the final dataset.

In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from lightgbm import LGBMRegressor

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / '0_data' / 'outputs' / 'combined_data.parquet').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / '0_data' / 'outputs' / 'combined_data.parquet').is_file():
    raise FileNotFoundError('Run this notebook from the repository root or a project subdirectory.')

SPLIT_SCHEME = 'stereo_cleaned'  # folder written by 1_data_split/data_split.ipynb
SPLIT_DIR = PROJECT_ROOT / '1_data_split' / 'outputs' / SPLIT_SCHEME
INPUT_PATHS = {name: SPLIT_DIR / f'{name}.parquet' for name in ['train', 'validation', 'test']}
SPLIT_SUMMARY_PATH = SPLIT_DIR / 'split_summary.csv'
COMBINED_DATA_PATH = PROJECT_ROOT / '0_data' / 'outputs' / 'combined_data.parquet'
OUTPUT_ROOT = PROJECT_ROOT / '2_feature_engineering' / 'outputs' / SPLIT_SCHEME
VALIDATION_REPORT_DIR = OUTPUT_ROOT / 'validation_reports'
SELECTED_DIR = OUTPUT_ROOT / 'selected_features'
REPRESENTATION_CACHE_DIR = PROJECT_ROOT / '2_feature_engineering' / 'outputs' / 'representation_cache'
for directory in [OUTPUT_ROOT, VALIDATION_REPORT_DIR, SELECTED_DIR, REPRESENTATION_CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Edit this list to change the generated representations. Optional blocks require their packages.
REPRESENTATIONS = ['rdkit2d', 'maccs', 'ecfp4_bit', 'chemberta2_mtr']
USE_GRAPH_EMBEDDINGS = True
GRAPH_EMB_PATH = PROJECT_ROOT / '0_data' / 'outputs' / 'combined_data_graph_embeddings.csv.gz'
FEATURE_BLOCKS = REPRESENTATIONS + (['graph_emb'] if USE_GRAPH_EMBEDDINGS else [])

RANDOM_STATE = 2026
MP_COLUMN = 'MP'
HIGH_MP_THRESHOLD_C = 250.0
BINARY_VARIANCE_THRESHOLD = 0.01
CONTINUOUS_VARIANCE_THRESHOLD = 1e-12
CORRELATION_THRESHOLD = 0.90
CORRELATION_RULE = 'greedy_abs_target_pearson'
MI_NEIGHBORS = 3
NAN_POLICY = 'drop_high_missing_columns_and_rare_rows'
RARE_MISSING_FRACTION = 0.0001  # 0.01% of valid training compounds
# Fixed LightGBM settings used only to compare candidate K values on the validation set.
K_SELECTION_LGBM_PARAMS = {'n_estimators': 300, 'learning_rate': 0.05, 'num_leaves': 31}

sys.path.insert(0, str(PROJECT_ROOT / '2_feature_engineering'))
import feature_representations
from feature_representations import generate_features, load_graph_embeddings

def file_sha256(path, chunk_size=1 << 20):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

print('Split scheme:', SPLIT_SCHEME)
print('Feature blocks:', FEATURE_BLOCKS)
print('Split directory:', SPLIT_DIR.relative_to(PROJECT_ROOT))
print('Output directory:', OUTPUT_ROOT.relative_to(PROJECT_ROOT))
print('Representation cache:', REPRESENTATION_CACHE_DIR.relative_to(PROJECT_ROOT))

Split scheme: stereo_cleaned
Feature blocks: ['rdkit2d', 'maccs', 'ecfp4_bit', 'chemberta2_mtr', 'graph_emb']
Split directory: 1_data_split/outputs/stereo_cleaned
Output directory: 2_feature_engineering/outputs/stereo_cleaned
Representation cache: 2_feature_engineering/outputs/representation_cache


## Load the fixed split

The split files are loaded without using the test labels for any selection decision. The split must have been created from the current `combined_data.parquet`; otherwise the notebook stops and `data_split.ipynb` must be rerun.

In [2]:
split_summary = pd.read_csv(SPLIT_SUMMARY_PATH)
dataset_sha256 = file_sha256(COMBINED_DATA_PATH)
if set(split_summary['Input_SHA256']) != {dataset_sha256}:
    raise ValueError(f'{SPLIT_DIR.name} was created from a different combined_data.parquet. Rerun data_split.ipynb.')

split_data = {}
for split_name, path in INPUT_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(f'Missing split file: {path}. Run data_split.ipynb first.')
    split_data[split_name] = pd.read_parquet(path)
    print(split_name, split_data[split_name].shape)

required_columns = {'SMILES', MP_COLUMN, 'MP_class', 'Split'}
for split_name, frame in split_data.items():
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise ValueError(f'{split_name} is missing columns: {missing}')
    assert frame['SMILES'].is_unique
    assert frame[MP_COLUMN].notna().all()

split_data['train']['MP_class'].value_counts(normalize=True)

train (166280, 21)
validation (55427, 21)
test (55427, 21)


MP_class
low     0.922745
high    0.077255
Name: proportion, dtype: float64

## Generate the selected representations

Representation generation is unsupervised: it uses SMILES only and does not fit on MP. Each representation is stored once per compound in `REPRESENTATION_CACHE_DIR/<representation>.parquet` together with its validation report, keyed by the input SMILES; only compounds missing from the cache are generated and appended. Each split's block is then looked up by SMILES and must align exactly with the split rows. Invalid SMILES and descriptor-specific calculation failures are represented as missing values and handled according to `NAN_POLICY`. ChemBERTa token truncation is recorded in its validation report.

In [3]:
# ChemBERTa2 uses CUDA when available, then Apple MPS, then CPU.
CHEMBERTA_DEVICE = None
CHEMBERTA_BATCH_SIZE = 32
CHEMBERTA_PROGRESS_EVERY = 25
CACHE_KEY = 'Input_SMILES'

def update_representation_cache(representation, smiles):
    # Return the cached table for `representation`, generating and appending any missing SMILES.
    cache_path = REPRESENTATION_CACHE_DIR / f'{representation}.parquet'
    cache = pd.read_parquet(cache_path) if cache_path.is_file() else None
    cached = set() if cache is None else set(cache[CACHE_KEY])
    missing = [value for value in pd.unique(smiles) if value not in cached]
    if missing:
        print(f'Generating {representation} for {len(missing):,} uncached compounds...', flush=True)
        generated, report = generate_features(
            pd.DataFrame({'SMILES': missing}),
            representations=[representation],
            strict=False,
            chemberta_batch_size=CHEMBERTA_BATCH_SIZE,
            chemberta_device=CHEMBERTA_DEVICE,
            chemberta_progress=True,
            chemberta_progress_every=CHEMBERTA_PROGRESS_EVERY,
        )
        new_rows = pd.concat([report.reset_index(drop=True), generated.drop(columns='SMILES')], axis=1)
        cache = new_rows if cache is None else pd.concat([cache, new_rows], ignore_index=True)
        if not cache[CACHE_KEY].is_unique:
            raise ValueError(f'Duplicate SMILES in the {representation} cache.')
        cache.to_parquet(cache_path, index=False)
        print(f'Saved {cache_path.relative_to(PROJECT_ROOT)} ({len(cache):,} compounds)', flush=True)
    else:
        print(f'Loaded cached {representation} ({len(cache):,} compounds)', flush=True)
    return cache.set_index(CACHE_KEY)

all_smiles = pd.concat([frame['SMILES'] for frame in split_data.values()], ignore_index=True)
blocks = {split_name: [] for split_name in split_data}
reports = {}
canonical_smiles = {}
for representation in REPRESENTATIONS:
    cache = update_representation_cache(representation, all_smiles)
    feature_columns = [column for column in cache.columns if column.startswith(f'{representation}_')]
    report_columns = [column for column in cache.columns if column not in set(feature_columns)]
    for split_name, frame in split_data.items():
        rows = cache.reindex(frame['SMILES'])
        if rows[feature_columns[0]].index.tolist() != frame['SMILES'].tolist() or rows['Status'].isna().any():
            raise ValueError(f'{representation} cache does not cover every {split_name} compound.')
        smiles_out = rows['Canonical_SMILES'].tolist()
        if split_name not in canonical_smiles:
            canonical_smiles[split_name] = smiles_out
            reports[split_name] = rows[report_columns].reset_index()
        elif smiles_out != canonical_smiles[split_name]:
            raise ValueError(f'Canonical SMILES mismatch for {split_name}/{representation}.')
        blocks[split_name].append(rows[feature_columns].reset_index(drop=True))
        print(f'  {split_name}: {len(feature_columns):,} {representation} features', flush=True)
    del cache

feature_data = {}
feature_column_sets = {}
for split_name, frame in split_data.items():
    combined = pd.concat([frame.reset_index(drop=True), *blocks[split_name]], axis=1)
    combined['SMILES'] = np.asarray(canonical_smiles[split_name], dtype=object)
    feature_data[split_name] = combined
    feature_column_sets[split_name] = [column for column in combined.columns if column not in frame.columns]
    reports[split_name].to_csv(VALIDATION_REPORT_DIR / f'{split_name}_validation_report.csv', index=False)
    print(f'Completed {split_name}: {combined.shape}; feature columns: {len(feature_column_sets[split_name]):,}', flush=True)
del blocks

feature_cols = feature_column_sets['train']
if not all(columns == feature_cols for columns in feature_column_sets.values()):
    raise ValueError('Representation feature columns differ across train, validation, and test.')
print(f'Common generated features: {len(feature_cols):,}')

Loaded cached rdkit2d (304,508 compounds)
  train: 217 rdkit2d features
  validation: 217 rdkit2d features
  test: 217 rdkit2d features
Loaded cached maccs (304,508 compounds)
  train: 167 maccs features
  validation: 167 maccs features
  test: 167 maccs features
Loaded cached ecfp4_bit (304,508 compounds)
  train: 2,048 ecfp4_bit features
  validation: 2,048 ecfp4_bit features
  test: 2,048 ecfp4_bit features
Loaded cached chemberta2_mtr (304,508 compounds)
  train: 384 chemberta2_mtr features
  validation: 384 chemberta2_mtr features
  test: 384 chemberta2_mtr features
Completed train: (166280, 2837); feature columns: 2,816
Completed validation: (55427, 2837); feature columns: 2,816
Completed test: (55427, 2837); feature columns: 2,816
Common generated features: 2,816


## Add the precomputed graph embeddings

The 512-dimensional graph-neural-network embeddings are looked up by SMILES in `GRAPH_EMB_PATH` and appended after the generated representations. Every compound must have an embedding. The embedding file is hashed into `feature_engineering_config.json`.

In [4]:
if USE_GRAPH_EMBEDDINGS:
    for split_name, frame in feature_data.items():
        embeddings, report = load_graph_embeddings(split_data[split_name]['SMILES'], path=GRAPH_EMB_PATH, strict=True)
        if embeddings['SMILES'].tolist() != frame['SMILES'].tolist():
            raise ValueError(f'Graph-embedding rows do not align with the {split_name} features.')
        feature_data[split_name] = pd.concat([frame, embeddings.drop(columns='SMILES')], axis=1)
        print(f'{split_name}: added {embeddings.shape[1] - 1} graph-embedding features for {len(embeddings):,} compounds')
    graph_columns = [column for column in feature_data['train'].columns if column.startswith('graph_emb_')]
    feature_cols = feature_cols + graph_columns
print(f'Features before filtering: {len(feature_cols):,}')

train: added 512 graph-embedding features for 166,280 compounds
validation: added 512 graph-embedding features for 55,427 compounds
test: added 512 graph-embedding features for 55,427 compounds
Features before filtering: 3,328


## Apply the configured missing-value policy

All missing, infinite, and non-numeric generated feature values are normalized to `NaN`. Columns with missingness above the configured 0.01% training threshold are removed, while rows affected by rarer missing values are removed. All column decisions are fitted on valid training rows only and audited.

In [5]:
if NAN_POLICY != 'drop_high_missing_columns_and_rare_rows':
    raise ValueError(f'Unsupported NAN_POLICY: {NAN_POLICY}')

# Normalize all generated feature problems to NaN before calculating missingness.
for split_name, frame in feature_data.items():
    frame[feature_cols] = frame[feature_cols].apply(pd.to_numeric, errors='coerce')
    frame[feature_cols] = frame[feature_cols].replace([np.inf, -np.inf], np.nan)
    feature_data[split_name] = frame

# Invalid SMILES invalidate the complete molecular row and are excluded before
# feature-level missingness decisions so they do not make every column appear missing.
invalid_smiles_audit = {}
for split_name, frame in feature_data.items():
    invalid_smiles = frame['SMILES'].isna()
    invalid_smiles_audit[split_name] = int(invalid_smiles.sum())
    feature_data[split_name] = frame.loc[~invalid_smiles].reset_index(drop=True)

train_valid = feature_data['train']
missing_train = train_valid[feature_cols].isna().sum()
rare_missing_limit = max(1, int(np.ceil(RARE_MISSING_FRACTION * len(train_valid))))
high_missing_columns = missing_train[missing_train > rare_missing_limit].index.tolist()
rare_missing_columns = missing_train[(missing_train > 0) & (missing_train <= rare_missing_limit)].index.tolist()

feature_missingness_audit = pd.DataFrame({'Feature': feature_cols})
feature_missingness_audit['Representation'] = feature_missingness_audit['Feature'].map(
    lambda column: next(block for block in FEATURE_BLOCKS if column.startswith(f'{block}_')))
feature_missingness_audit['Train_missing_values'] = missing_train.reindex(feature_cols).to_numpy()
feature_missingness_audit['Train_missing_fraction'] = feature_missingness_audit['Train_missing_values'] / len(train_valid)
feature_missingness_audit['Action'] = 'keep'
feature_missingness_audit.loc[feature_missingness_audit['Feature'].isin(high_missing_columns), 'Action'] = 'drop_column'
feature_missingness_audit.loc[feature_missingness_audit['Feature'].isin(rare_missing_columns), 'Action'] = 'drop_affected_rows'
feature_missingness_audit.to_csv(OUTPUT_ROOT / 'feature_missingness_audit.csv', index=False)

feature_cols = [column for column in feature_cols if column not in set(high_missing_columns)]
for split_name, frame in feature_data.items():
    feature_data[split_name] = frame.drop(columns=high_missing_columns).copy()

nan_audit = []
for split_name, frame in feature_data.items():
    rare_missing_rows = frame[rare_missing_columns].isna().any(axis=1) if rare_missing_columns else pd.Series(False, index=frame.index)
    remaining_feature_nan = frame[feature_cols].isna().any(axis=1)
    bad_rows = rare_missing_rows | remaining_feature_nan
    nan_audit.append({'Split': split_name, 'Rows_before_feature_filter': len(frame), 'Rows_with_invalid_SMILES': invalid_smiles_audit[split_name], 'Rows_with_rare_NaN': int(rare_missing_rows.sum()), 'Rows_with_remaining_feature_NaN': int(remaining_feature_nan.sum()), 'Rows_dropped': int(bad_rows.sum()), 'Rows_after': int((~bad_rows).sum())})
    feature_data[split_name] = frame.loc[~bad_rows].reset_index(drop=True)

nan_audit = pd.DataFrame(nan_audit)
nan_audit.to_csv(OUTPUT_ROOT / 'nan_audit.csv', index=False)
print(f'Rare missingness threshold: {RARE_MISSING_FRACTION:.4%} ({rare_missing_limit} training compounds)')
print(f'Dropped columns: {len(high_missing_columns):,}')
display(feature_missingness_audit['Action'].value_counts().rename_axis('Action').to_frame('Feature_count'))
display(nan_audit)

Rare missingness threshold: 0.0100% (17 training compounds)
Dropped columns: 12


,Feature_count
Action,
keep,3316
drop_column,12


,Split,Rows_before_feature_filter,Rows_with_invalid_SMILES,Rows_with_rare_NaN,Rows_with_remaining_feature_NaN,Rows_dropped,Rows_after
0,train,166280,0,0,0,0,166280
1,validation,55427,0,0,0,0,55427
2,test,55427,0,0,0,0,55427


## Define feature blocks and fit the variance and correlation filters on training data

Binary blocks (MACCS keys and ECFP4 bits) are identified by their representation prefixes; all other representations (RDKit descriptors, ChemBERTa and graph embeddings) are treated as continuous. Each block has its own variance threshold, and the correlation filter compares features only within a block.

The correlation filter is target-guided. Features are visited in order of decreasing absolute Pearson correlation with MP, so a strongly negative correlation counts the same as a strongly positive one. A feature is kept only if its absolute correlation with every feature already kept is at most `CORRELATION_THRESHOLD`; otherwise it is dropped in favor of the kept feature it is most correlated with. Because features are compared only with kept features, a feature is never removed because of a feature that was itself removed. Ties in target correlation follow the column order (the `FEATURE_BLOCKS` order). Both filters are fitted on training data only and then applied to validation and test. `correlation_filter_audit.csv` records, for every feature, its target correlation and, if dropped, the feature that replaced it.

In [6]:
NON_FEATURE_COLS = [
    'SMILES', 'MP', 'Source', 'Type', 'MP_label', 'MP_class', 'Split',
    'MP_mean', 'MP_std', 'MP_mad', 'MP_min', 'MP_max', 'MP_range',
    'N_MP_values', 'N_distinct_MP', 'N_sources', 'MP_quality',
    'Consensus_method', 'Has_excluded_values', 'N_excluded_values',
    'All_MP_range', 'ContainsHalogen', 'MW', 'SMILES_nonisomeric'
]

binary_prefixes = ('maccs_', 'ecfp4_bit_')
binary_features = [column for column in feature_cols if column.startswith(binary_prefixes)]
continuous_features = [column for column in feature_cols if column not in binary_features]
REPRESENTATION_PREFIXES = {block: f'{block}_' for block in FEATURE_BLOCKS}

def representation_of(column):
    return next(name for name, prefix in REPRESENTATION_PREFIXES.items() if column.startswith(prefix))

def correlation_keep_columns(frame, columns, threshold, target, block_name):
    # Greedy filter: visit features by decreasing |Pearson r| with the target and keep a
    # feature only if |r| with every already-kept feature is <= threshold.
    columns = list(columns)
    if not columns:
        return [], pd.DataFrame()
    values = frame[columns].to_numpy(dtype=np.float32)
    values -= values.mean(axis=0)
    std = values.std(axis=0)
    if not (std > 0).all():
        raise ValueError(f'Constant columns reached the {block_name} correlation filter.')
    values /= std
    y = target.to_numpy(dtype=np.float64)
    y = ((y - y.mean()) / y.std()).astype(np.float32)
    n_rows = len(values)
    target_r = (values.T @ y).astype(np.float64) / n_rows
    abs_corr = np.abs(values.T @ values) / n_rows
    del values

    order = np.lexsort((np.arange(len(columns)), -np.abs(target_r)))
    kept = []
    dropped_in_favor_of = [None] * len(columns)
    abs_r_with_keeper = np.full(len(columns), np.nan)
    for position in order:
        if kept:
            partner_r = abs_corr[position, kept]
            best = int(np.argmax(partner_r))
            if partner_r[best] > threshold:
                dropped_in_favor_of[position] = columns[kept[best]]
                abs_r_with_keeper[position] = float(partner_r[best])
                continue
        kept.append(position)

    target_rank = np.empty(len(columns), dtype=int)
    target_rank[order] = np.arange(1, len(columns) + 1)
    kept_set = set(kept)
    audit = pd.DataFrame({
        'Feature': columns,
        'Block': block_name,
        'Representation': [representation_of(column) for column in columns],
        'Target_r': target_r,
        'Abs_target_r': np.abs(target_r),
        'Target_rank': target_rank,
        'Kept': [index in kept_set for index in range(len(columns))],
        'Dropped_in_favor_of': dropped_in_favor_of,
        'Abs_r_with_keeper': abs_r_with_keeper,
    })
    return [columns[index] for index in sorted(kept)], audit

def fit_train_filters(frame, target):
    binary_variance = frame[binary_features].var(axis=0, ddof=0)
    continuous_variance = frame[continuous_features].var(axis=0, ddof=0)
    binary_var_keep = binary_variance[binary_variance > BINARY_VARIANCE_THRESHOLD].index.tolist()
    continuous_var_keep = continuous_variance[continuous_variance > CONTINUOUS_VARIANCE_THRESHOLD].index.tolist()
    binary_corr_keep, binary_audit = correlation_keep_columns(frame, binary_var_keep, CORRELATION_THRESHOLD, target, 'binary')
    continuous_corr_keep, continuous_audit = correlation_keep_columns(frame, continuous_var_keep, CORRELATION_THRESHOLD, target, 'continuous')
    correlation_audit = pd.concat([binary_audit, continuous_audit], ignore_index=True)
    return binary_var_keep, continuous_var_keep, binary_corr_keep, continuous_corr_keep, correlation_audit

train_data = feature_data['train']
(train_binary_var_keep, train_continuous_var_keep,
 train_binary_keep, train_continuous_keep, correlation_audit) = fit_train_filters(train_data, train_data[MP_COLUMN])
correlation_audit.sort_values(['Block', 'Target_rank']).to_csv(OUTPUT_ROOT / 'correlation_filter_audit.csv', index=False)
train_filtered_features = train_binary_keep + train_continuous_keep
filter_summary = pd.DataFrame({
    'Block': ['binary', 'continuous'],
    'Original': [len(binary_features), len(continuous_features)],
    'After_variance': [len(train_binary_var_keep), len(train_continuous_var_keep)],
    'After_correlation': [len(train_binary_keep), len(train_continuous_keep)],
})
display(filter_summary)
print(f'Retained after variance/correlation filtering: {len(train_filtered_features):,}')

,Block,Original,After_variance,After_correlation
0,binary,2215,883,851
1,continuous,1101,1098,839


Retained after variance/correlation filtering: 1,690


## Mutual-information ranking and validation-based K selection

Mutual information is calculated on the training data only. Candidate Top-K subsets, including the full retained set, are then trained on the training data and evaluated on the fixed validation set, so the step may keep every retained feature if that validates best. The test set is not accessed in this section.

In [7]:
X_train_filtered = train_data[train_filtered_features]
y_train = train_data[MP_COLUMN]
discrete_mask = np.array([column in set(train_binary_keep) for column in train_filtered_features])
mi_scores = mutual_info_regression(X_train_filtered, y_train, discrete_features=discrete_mask, random_state=RANDOM_STATE, n_neighbors=MI_NEIGHBORS)
mi_ranking = pd.DataFrame({'Feature': train_filtered_features, 'Mutual_information': mi_scores}).sort_values(['Mutual_information', 'Feature'], ascending=[False, True], kind='stable').reset_index(drop=True)
mi_ranking.to_csv(OUTPUT_ROOT / 'mutual_information_ranking.csv', index=False)

n_retained = len(mi_ranking)
candidate_k = [20, 50, 100, 200, int(round(0.25 * n_retained)), int(round(0.50 * n_retained)), int(round(0.70 * n_retained)), n_retained]
candidate_k = sorted({max(1, min(n_retained, k)) for k in candidate_k})

X_validation = feature_data['validation']
y_validation = X_validation[MP_COLUMN]
k_results = []
for k in candidate_k:
    selected = mi_ranking.head(k)['Feature'].tolist()
    model = LGBMRegressor(**K_SELECTION_LGBM_PARAMS, random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1)
    model.fit(train_data[selected], y_train)
    prediction = model.predict(X_validation[selected])
    k_results.append({'K': k, 'Validation_RMSE': np.sqrt(mean_squared_error(y_validation, prediction)), 'Validation_MAE': mean_absolute_error(y_validation, prediction), 'Validation_R2': r2_score(y_validation, prediction)})

k_results = pd.DataFrame(k_results).sort_values('K').reset_index(drop=True)
k_results.to_csv(OUTPUT_ROOT / 'k_validation_results.csv', index=False)
best_k = int(k_results.loc[k_results['Validation_RMSE'].idxmin(), 'K'])
print(f'Selected K based on validation RMSE: {best_k}')
display(k_results)

Selected K based on validation RMSE: 1690


,K,Validation_RMSE,Validation_MAE,Validation_R2
0,20,45.216791,35.426514,0.486430
1,50,43.599843,34.005587,0.522503
2,100,42.408424,32.862478,0.548243
3,200,41.660138,32.265904,0.564045
4,422,41.222029,31.838614,0.573166
5,845,40.470002,31.158511,0.588597
6,1183,40.133035,30.843952,0.595420
7,1690,40.035155,30.756253,0.597391


## Finalize and save the selected feature set

After K is selected, the corresponding train-fitted feature subset is finalized and saved for downstream model development. No final model is fit and the test set is not evaluated in this notebook.

In [8]:
# The selected list comes directly from the train-fitted MI ranking used for K selection.
# The provisional LightGBM models above only compare candidate K values; they are not final models.
final_features = mi_ranking.head(best_k)['Feature'].tolist()
selected_feature_table = mi_ranking.head(best_k).copy()
selected_feature_table.insert(0, 'Rank', np.arange(1, len(selected_feature_table) + 1))
selected_feature_table.to_csv(OUTPUT_ROOT / 'selected_features.csv', index=False)

def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

for split_name, frame in feature_data.items():
    output_columns = [column for column in NON_FEATURE_COLS if column in frame.columns] + final_features
    frame[output_columns].to_parquet(SELECTED_DIR / f'{split_name}_selected_features.parquet', index=False)

config = {
    'split_scheme': SPLIT_SCHEME,
    'dataset': str(COMBINED_DATA_PATH.relative_to(PROJECT_ROOT)),
    'dataset_sha256': dataset_sha256,
    'feature_blocks': FEATURE_BLOCKS,
    'random_state': RANDOM_STATE,
    'mp_column': MP_COLUMN,
    'high_mp_threshold_C': HIGH_MP_THRESHOLD_C,
    'binary_variance_threshold': BINARY_VARIANCE_THRESHOLD,
    'continuous_variance_threshold': CONTINUOUS_VARIANCE_THRESHOLD,
    'correlation_threshold': CORRELATION_THRESHOLD,
    'correlation_rule': CORRELATION_RULE,
    'mi_neighbors': MI_NEIGHBORS,
    'nan_policy': NAN_POLICY,
    'rare_missing_fraction': RARE_MISSING_FRACTION,
    'selection_fit_data': 'train_only',
    'k_selection_data': 'fixed_validation',
    'candidate_K': candidate_k,
    'k_selection_model': {'type': 'LGBMRegressor', **K_SELECTION_LGBM_PARAMS, 'random_state': RANDOM_STATE},
    'selected_K': best_k,
    'selected_feature_count': len(final_features),
    'test_used': False,
    'final_model_fit': False,
    'representation_settings': {
        'rdkit2d': {'source': 'rdkit.Chem.Descriptors.descList',
                    'descriptors_generated': sum(column.startswith('rdkit2d_') for column in feature_column_sets['train'])},
        'maccs': {'n_bits': feature_representations.MACCS_BITS},
        'ecfp4_bit': {'radius': feature_representations.DEFAULT_ECFP_RADIUS,
                      'n_bits': feature_representations.DEFAULT_ECFP_BITS,
                      'use_chirality': feature_representations.DEFAULT_ECFP_USE_CHIRALITY,
                      'use_bond_types': feature_representations.DEFAULT_ECFP_USE_BOND_TYPES},
        'chemberta2_mtr': {'checkpoint': feature_representations.DEFAULT_CHEMBERTA_CHECKPOINT,
                           'max_length': feature_representations.DEFAULT_CHEMBERTA_MAX_LENGTH,
                           'pooling': 'attention-mask mean of the final hidden layer',
                           'fine_tuned': False},
    },
    # Representations are read from the shared cache, which may have been generated in an earlier session;
    # the cache hashes identify the exact values used, and the versions below are those of this run.
    'representation_cache_sha256': {representation: file_sha256(REPRESENTATION_CACHE_DIR / f'{representation}.parquet')
                                    for representation in REPRESENTATIONS},
    'package_versions': {'python': sys.version.split()[0],
                         **{name: package_version(name) for name in ['rdkit', 'numpy', 'pandas', 'scikit-learn',
                                                                     'lightgbm', 'torch', 'transformers']}},
}
if USE_GRAPH_EMBEDDINGS:
    config['graph_embedding_source'] = str(GRAPH_EMB_PATH.relative_to(PROJECT_ROOT))
    config['graph_embedding_sha256'] = file_sha256(GRAPH_EMB_PATH)
    config['graph_embedding_dimension'] = len(graph_columns)
    config['graph_embedding_provenance'] = 'external graph model, not trained with MP labels; independent of this split'
(OUTPUT_ROOT / 'feature_engineering_config.json').write_text(json.dumps(config, indent=2))
print(f'Finalized feature set: K={best_k} ({len(final_features):,} features)')
print('Saved feature names and selected train/validation/test tables for downstream model development.')
print(f'Outputs written to: {OUTPUT_ROOT.relative_to(PROJECT_ROOT)}')

Finalized feature set: K=1690 (1,690 features)
Saved feature names and selected train/validation/test tables for downstream model development.
Outputs written to: 2_feature_engineering/outputs/stereo_cleaned


## Representation-level filter summary

This audit is fitted on the training data. `Original` means features available after the configured missing-value handling and before variance filtering. The variance and correlation columns show how many features from each representation remain after each filter, and `Selected` shows how many are in the final MI/K-selected set.

In [9]:
REPRESENTATION_LABELS = {
    'rdkit2d': 'RDKit2D',
    'maccs': 'MACCS',
    'ecfp4_bit': 'ECFP4_bit',
    'ecfp4_count': 'ECFP4_count',
    'chemberta2_mtr': 'ChemBERTa2_MTR',
    'graph_emb': 'GraphEmb',
}

variance_set = set(train_binary_var_keep + train_continuous_var_keep)
correlation_set = set(train_filtered_features)
selected_set = set(final_features)

representation_rows = []
for representation, prefix in REPRESENTATION_PREFIXES.items():
    original = [column for column in feature_cols if column.startswith(prefix)]
    after_variance = [column for column in original if column in variance_set]
    after_correlation = [column for column in after_variance if column in correlation_set]
    selected = [column for column in after_correlation if column in selected_set]
    representation_rows.append({
        'Representation': REPRESENTATION_LABELS.get(representation, representation),
        'Original': len(original),
        'After_variance': len(after_variance),
        'After_correlation': len(after_correlation),
        'Selected': len(selected),
        'Removed_by_variance': len(original) - len(after_variance),
        'Removed_by_correlation': len(after_variance) - len(after_correlation),
        'Retained_after_correlation_percent': round(100 * len(after_correlation) / len(original), 2) if original else 0.0,
    })

representation_filter_summary = pd.DataFrame(representation_rows)
representation_filter_summary.to_csv(OUTPUT_ROOT / 'representation_filter_summary.csv', index=False)
display(representation_filter_summary)
print(f'Saved representation filter summary to {(OUTPUT_ROOT / "representation_filter_summary.csv").relative_to(PROJECT_ROOT)}')

,Representation,Original,After_variance,After_correlation,Selected,Removed_by_variance,Removed_by_correlation,Retained_after_correlation_percent
0,RDKit2D,205,202,166,166,3,36,80.98
1,MACCS,167,141,120,120,26,21,71.86
2,ECFP4_bit,2048,742,731,731,1306,11,35.69
3,ChemBERTa2_MTR,384,384,380,380,0,4,98.96
4,GraphEmb,512,512,293,293,0,219,57.23


Saved representation filter summary to 2_feature_engineering/outputs/stereo_cleaned/representation_filter_summary.csv
